# Read And Merge Experiment Config From Comet

This notebook supports a legacy-to-current workflow:

1. Read a config reconstructed from a Comet experiment key.
2. Load a current config from local YAML files.
3. Replace selected sections such as `train` or `mix_data` with the local current versions.
4. Optionally edit a few fields such as the model latent dimension.
5. Save the merged config as a new YAML file.

In [27]:
from copy import deepcopy
from dataclasses import asdict, is_dataclass
from importlib import reload
from pathlib import Path
from pprint import pprint

import comet_ml

import pff.training.basic_experiment as basic_experiment_module
import pff.training.utils as training_utils_module
from pff import config_dir
from pff.models import get_model_config

# Reload project modules so notebook sessions do not keep stale definitions.
reload(training_utils_module)
reload(basic_experiment_module)

BasicLightningExperiment = basic_experiment_module.BasicLightningExperiment
_resolve_comet_key = basic_experiment_module._resolve_comet_key
parse_comet_parameters_summary = training_utils_module.parse_comet_parameters_summary
save_experiment_config_yaml = training_utils_module.save_experiment_config_yaml


In [52]:
# Comet experiment to recover.
# You can pass either the raw experiment key or the full Comet URL.
experiment_key = "c0da64e4ae47472a90ad1677ef5292da"

# Current local config to use as the source of updated sections.
local_config_path = Path(config_dir) / "experiment_configs" / "UAI" / "aicme-t-pk" / "base.yaml"

# Sections taken from the local config and copied into the Comet-derived one.
sections_to_replace = [
    "train",
    "mix_data",
]

# Optional output directory for the merged config.


output_dir = Path(config_dir) / "experiment_configs" / "UAI" / "aicme-t-pk-success"

# Optional: only needed for the full experiment restoration path below.
checkpoint_type = "best"


In [53]:
def inspect_config(exp_config):
    """Pretty-print a config dataclass for inspection."""

    if is_dataclass(exp_config):
        pprint(asdict(exp_config))
    else:
        pprint(exp_config)


def load_config_from_comet(experiment_key: str):
    """Load only the experiment config reconstructed from Comet parameters."""

    if experiment_key is None:
        raise ValueError("experiment_key must be provided.")

    normalized_experiment_key = str(experiment_key).strip().rstrip("/").split("/")[-1]
    if (
        not normalized_experiment_key
        or normalized_experiment_key == "PASTE_COMET_EXPERIMENT_KEY_OR_URL_HERE"
    ):
        raise ValueError(
            "Please set experiment_key to a real Comet experiment key or URL before running this cell."
        )

    comet_api_key = _resolve_comet_key(None)
    if comet_api_key is None:
        raise ValueError(
            "No Comet API key was found. Set COMET_KEYS.txt or provide access in the notebook kernel."
        )

    api = comet_ml.API(api_key=comet_api_key)
    api_experiment = api.get_experiment_by_key(normalized_experiment_key)
    if api_experiment is None:
        raise ValueError(
            f"Comet experiment '{normalized_experiment_key}' was not found or is not accessible with the current API key. "
            "If you passed a URL, make sure it points to an experiment page. If you passed a key, verify it is the actual experiment id and that the current Comet account can read it."
        )

    parameters_summary = api_experiment.get_parameters_summary()
    exp_config = parse_comet_parameters_summary(parameters_summary)
    return exp_config, api_experiment, parameters_summary


def merge_config_sections(base_config, source_config, sections):
    """Copy selected top-level sections from ``source_config`` into ``base_config``."""

    merged_config = deepcopy(base_config)
    for section_name in sections:
        if not hasattr(source_config, section_name):
            raise AttributeError(f"source_config has no section '{section_name}'")
        setattr(merged_config, section_name, deepcopy(getattr(source_config, section_name)))
    return merged_config


In [54]:
comet_config, api_experiment, parameters_summary = load_config_from_comet(experiment_key)
local_config = get_model_config(str(local_config_path))

type(comet_config), type(local_config)


(pff.config_classes.node_pk_config.NodePKExperimentConfig,
 pff.config_classes.node_pk_config.NodePKExperimentConfig)

In [55]:
print("Comet config type:", type(comet_config))
print("Local config type:", type(local_config))
print("Comet latent dim:", comet_config.network.zi_latent_dim)
print("Local latent dim:", local_config.network.zi_latent_dim)
print("Comet train epochs:", comet_config.train.epochs)
print("Local train epochs:", local_config.train.epochs)


Comet config type: <class 'pff.config_classes.node_pk_config.NodePKExperimentConfig'>
Local config type: <class 'pff.config_classes.node_pk_config.NodePKExperimentConfig'>
Comet latent dim: 256
Local latent dim: 256
Comet train epochs: 100
Local train epochs: 100


In [56]:
# Full inspection if needed.
# inspect_config(comet_config)
# inspect_config(local_config)


## Merge legacy Comet config with current local sections

In [57]:
merged_config = merge_config_sections(
    base_config=comet_config,
    source_config=local_config,
    sections=sections_to_replace,
)

print("Merged train epochs:", merged_config.train.epochs)
print(
    "Merged validation scheduler present:",
    merged_config.train.callbacks_scheduler is not None,
)
print("Merged test datasets:", merged_config.mix_data.test_empirical_datasets)


Merged train epochs: 100
Merged validation scheduler present: True
Merged test datasets: ['cesarali/lenuzza-2016', 'cesarali/Indometacin', 'cesarali/Theophylline']


## Optional manual edits after the merge

In [58]:
edited_config = deepcopy(merged_config)

# Example change requested by you.
edited_config.network.zi_latent_dim = 128

# Optional bookkeeping updates for a future run or export.
edited_config.name_str = f"{edited_config.name_str}_latent128"
edited_config.experiment_indentifier = None
edited_config.experiment_dir = None

print("Edited latent dim:", edited_config.network.zi_latent_dim)
print("Edited name_str:", edited_config.name_str)


Edited latent dim: 128
Edited name_str: AICMEPK_latent128


In [59]:
#inspect_config(edited_config)


## Save merged or edited config to YAML

In [60]:
saved_yaml_path = save_experiment_config_yaml(
    exp_config=edited_config,
    experiment_dir=str(output_dir),
)
saved_yaml_path


'/home/cesarali/Pharma/pff/config_files/experiment_configs/UAI/aicme-t-pk-success/experiment_config.yaml'

In [61]:
output_dir

PosixPath('/home/cesarali/Pharma/pff/config_files/experiment_configs/UAI/aicme-t-pk-success')

## Optional: load the full experiment object

Use this only when you also want the restored checkpoint, model, and datamodule. For config migration alone, the cells above are enough.

In [ ]:
# Uncomment if you want the full experiment restoration path.
# experiment = BasicLightningExperiment.from_experiment_comet(
#     experiment_key=experiment_key,
#     map_location="cpu",
#     checkpoint_type=checkpoint_type,
# )
# experiment.exp_config


In [ ]:
# Optional cleanup for the Comet API handle.
api_experiment.end()
